# 🎬 MiniMax H3｜Google Colab Pro 圖生影片工作站（免 8188／免 403）
**操作方式：** Colab 選擇 GPU（建議 L4 24GB、A100 40/80GB）→ 按「執行階段 → 全部執行」→ 按畫面授權 Drive 並上傳圖片 → 自動生成 MP4。

- 使用 **MiniMax H3 FL2VA INT8**、原生聲音、官方 ComfyUI 節點。
- 不公開 ComfyUI 服務，不開 8188 代理網址；程式直接透過本機 API 生成，避開你遇到的 403。
- 首次需要從 Hugging Face 下載**約 37GB 以上**模型，亦需要足夠 CPU RAM/磁碟空間。執行時間和成功率取決於 Colab 分配的資源。
- **Colab Pro 不保證 A100 或特定 GPU**；T4 16GB 此設定不推薦。
- 本 Notebook 尚未在你的 Colab 執行階段做完整 GPU 推論驗證，發生錯誤時請貼出最後的錯誤區塊。
- 商用前請核對 MiniMax H3 的模型授權條款：[官方 ComfyUI 文件](https://docs.comfy.org/tutorials/video/minimax/minimax-h3)。


In [ ]:
# ====== ① 自訂設定（只需修改這一格） ======
PROMPT = """
Use the uploaded anime character as the exact identity reference. Preserve the face,
hair style, hair color, clothes and accessories. In a fantasy temple, the character
slowly draws a glowing sword and makes one clean diagonal slash.
Detailed Japanese anime cel shading, cinematic camera motion, smooth hair movement,
bright rim light, anatomically correct hands, consistent character identity.
Audio: soft wind, a clear metallic sword draw and one magical sword swoosh.
No subtitles, no letters, no logos, no extra limbs.
""".strip()

# 先做 5 秒測試：124 幀 / 24fps（H3 使用 17k+5 幀格）
FRAME_COUNT = 124              # 124≈5秒；243≈10秒；362≈15秒
WIDTH, HEIGHT = 448, 768       # 直式接近 9:16；均需是 32 的倍數
SEED = 42
STEPS = 20                     # 官方基本方案；較快的 Turbo 尚未啟用
SAVE_TO_GOOGLE_DRIVE = True
PERSIST_MODEL_TO_DRIVE = False # 避免 Drive 讀取速度過慢；模型在 Colab 暫存磁碟

assert FRAME_COUNT >= 5 and (FRAME_COUNT - 5) % 17 == 0, "幀數需符合 17k+5"
assert WIDTH % 32 == HEIGHT % 32 == 0, "長寬需為 32 的倍數"

import os, sys, subprocess, time, json, shutil, uuid, traceback
from pathlib import Path
import requests
from IPython.display import display, Video
import torch

if not torch.cuda.is_available():
    raise RuntimeError("尚未啟用 GPU！請選「執行階段 → 變更執行階段類型 → GPU」")
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / (1024**3)
print("GPU:", props.name, "| VRAM:", round(vram, 1), "GiB")
if vram < 20:
    raise RuntimeError("此流程為 L4 24GB／A100 設計。你的 GPU 低於 20GB，請改選 L4/A100。")
disk = shutil.disk_usage("/content").free / (1024**3)
print("可用磁碟:",round(disk,1),"GiB")
if disk < 48:
    raise RuntimeError("磁碟不足。請使用較大的 Colab 執行階段，至少預留 48GiB。")

BASE = Path("/content/ComfyUI")
DRIVE_OUTPUT = None
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT = Path("/content/drive/MyDrive/MiniMax_H3_Outputs")
    DRIVE_OUTPUT.mkdir(parents=True,exist_ok=True)
    print("Drive 輸出：",DRIVE_OUTPUT)


In [ ]:
# ====== ② 安裝官方 ComfyUI 與執行環境 ======
def run(args, cwd=None):
    print("執行:", " ".join(map(str,args)))
    subprocess.run(list(map(str,args)), cwd=str(cwd) if cwd else None, check=True)

if not (BASE/"main.py").exists():
    run(["git","clone","--depth","1","https://github.com/Comfy-Org/ComfyUI.git",BASE])
else:
    print("已找到 ComfyUI，略過 git clone。")

run([sys.executable,"-m","pip","install","-q","-r",BASE/"requirements.txt"])
run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub","hf_xet","requests","Pillow","imageio-ffmpeg"])

if not shutil.which("ffmpeg"):
    run(["apt-get","update","-qq"])
    run(["apt-get","install","-y","-qq","ffmpeg"])

for d in ["diffusion_models","text_encoders","vae","loras"]:
    (BASE/"models"/d).mkdir(parents=True,exist_ok=True)
(BASE/"input").mkdir(parents=True,exist_ok=True)
(BASE/"output").mkdir(parents=True,exist_ok=True)
print("✅ ComfyUI 已安裝")


In [ ]:
# ====== ③ 下載 MiniMax H3 圖生影片所需 4 個模型 ======
# 模型位置以 Comfy-Org 官方工作流程為準：
# https://docs.comfy.org/tutorials/video/minimax/minimax-h3-native
from huggingface_hub import hf_hub_download

REPO = "Comfy-Org/MiniMax-H3"
FILES = [
 "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
 "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
 "vae/minimax_h3_video_vae_int8_convrot.safetensors",
 "vae/minimax_h3_audio_vae_fp32.safetensors",
]
for f in FILES:
    dest = BASE/"models"/f
    if dest.exists() and dest.stat().st_size > 100*1024*1024:
        print("已存在：",f)
        continue
    print("正在下載：",f,flush=True)
    hf_hub_download(repo_id=REPO,filename=f,local_dir=str(BASE/"models"))
    if not dest.exists(): raise RuntimeError("模型下載未完成："+f)

print("✅ 必要模型全部就緒")


In [ ]:
# ====== ④ 在 Colab 背後啟動 ComfyUI，不透過網頁代理 ======
import urllib.request

PORT=8188
URL=f"http://127.0.0.1:{PORT}"
LOG=Path("/content/h3_comfyui.log")
def server_ready():
    try:
        return requests.get(URL+"/system_stats",timeout=3).status_code == 200
    except Exception:
        return False

if not server_ready():
    logfile = open(LOG,"a", buffering=1)
    cmd=[sys.executable,"main.py",
         "--listen","127.0.0.1","--port",str(PORT),
         "--lowvram","--disable-pinned-memory","--fp16-intermediates"]
    SERVER=subprocess.Popen(cmd,cwd=str(BASE),stdout=logfile,stderr=subprocess.STDOUT)

    for _ in range(180):
        if server_ready(): break
        if SERVER.poll() is not None:
            print(LOG.read_text(errors="replace")[-12000:])
            raise RuntimeError("ComfyUI 啟動失敗，請複製上述日誌")
        time.sleep(2)

if not server_ready():
    print(LOG.read_text(errors="replace")[-12000:])
    raise TimeoutError("ComfyUI 啟動逾時")

print("✅ ComfyUI 本機 API 正常。不需要開啟 8188 網頁！")


In [ ]:
# ====== ⑤ 上傳角色圖片（按「選擇檔案」） ======
from google.colab import files
from PIL import Image, ImageOps

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("你沒有上傳圖片，請重跑此儲存格")

fname, content = next(iter(uploaded.items()))
from io import BytesIO

img=Image.open(BytesIO(content)).convert("RGB")
# 第一幀會依畫布比例調整；先等比裁切避免拉長角色。
img = ImageOps.fit(img,(WIDTH,HEIGHT),method=Image.Resampling.LANCZOS,centering=(0.5,0.5))
IMAGE_NAME=f"h3_reference_{uuid.uuid4().hex[:8]}.png"
IMAGE_FILE=BASE/"input"/IMAGE_NAME
img.save(IMAGE_FILE)
print("✅ 已備妥 H3 首幀:",IMAGE_FILE)
display(img.resize((224,384)))


In [ ]:
# ====== ⑥ 呼叫官方 H3 ComfyUI API，圖生影片 + 原生音效 ======
# 完整 API-format 工作流程（不使用帶子圖的 UI JSON）。
# 節點與 Comfy-Org 官方 H3 I2V 工作流程一致。
def node(class_type, **inputs): return {"class_type":class_type,"inputs":inputs}

prompt={
 "6":node("UNETLoader",unet_name="minimax_h3_fl2va_pruned_int8_convrot.safetensors",weight_dtype="default"),
 "13":node("CLIPLoader",clip_name="qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",type="minimax",device="default"),
 "11":node("VAELoader",vae_name="minimax_h3_video_vae_int8_convrot.safetensors"),
 "24":node("VAELoader",vae_name="minimax_h3_audio_vae_fp32.safetensors"),
 "30":node("LoadImage",image=IMAGE_NAME),
 "104":node("MiniMaxH3ImageToVideo",clip=["13",0],vae=["11",0],first_frame=["30",0],
            width=WIDTH,height=HEIGHT,length=FRAME_COUNT,prompt=PROMPT),
 "16":node("BasicGuider",model=["6",0],conditioning=["104",0]),
 "17":node("KSamplerSelect",sampler_name="res_multistep"),
 "9":node("BasicScheduler",model=["6",0],scheduler="simple",steps=STEPS,denoise=1.0),
 "15":node("RandomNoise",noise_seed=SEED),
 "14":node("SamplerCustomAdvanced",noise=["15",0],guider=["16",0],sampler=["17",0],
           sigmas=["9",0],latent_image=["104",1]),
 "10":node("VAEDecode",samples=["14",0],vae=["11",0]),
 "23":node("VAEDecodeAudio",samples=["14",0],vae=["24",0]),
 "91":node("CreateVideo",images=["10",0],audio=["23",0],fps=24),
 "92":node("SaveVideo",video=["91",0],filename_prefix="MiniMaxH3/Anime_H3",
           format="auto",codec="auto"),
}

started=time.time()
print(f"排入生成：{WIDTH}×{HEIGHT}、{FRAME_COUNT} 幀、{STEPS} 步",flush=True)
r=requests.post(URL+"/prompt",json={"prompt":prompt},timeout=60)
if r.status_code!=200:
    print("工作流程驗證失敗：",r.text[:10000])
    raise RuntimeError("提交 H3 工作流程失敗。請貼上上面的錯誤訊息。")
prompt_id=r.json()["prompt_id"]
print("✅ 工作流程已提交：",prompt_id,flush=True)

history=None
deadline=time.time()+120*60
while time.time()<deadline:
    time.sleep(15)
    resp=requests.get(URL+"/history/"+prompt_id,timeout=30)
    resp.raise_for_status()
    h=resp.json()
    if prompt_id in h:
        history=h[prompt_id]
        status=history.get("status",{})
        if status.get("status_str")=="error":
            print(json.dumps(status,ensure_ascii=False,indent=2)[-12000:])
            raise RuntimeError("H3 推論失敗：詳見上方的 ComfyUI 錯誤")
        if status.get("completed") or status.get("status_str")=="success":
            break
    print("H3 正在生成中；已經過",int((time.time()-started)//60),"分鐘",flush=True)
else:
    raise TimeoutError("已達到此 Notebook 設定的最長輪詢時間")

if history is None: raise RuntimeError("找不到執行歷史記錄")

def output_records(x):
    if isinstance(x,dict):
        if "filename" in x: yield x
        for v in x.values(): yield from output_records(v)
    elif isinstance(x,list):
        for v in x: yield from output_records(v)

records=list(output_records(history.get("outputs",{})))
output_candidates=[]
for record in records:
    name=record.get("filename","")
    if name.lower().endswith((".mp4",".webm")):
        path=BASE/(record.get("type","output"))/record.get("subfolder","")/name
        if path.exists(): output_candidates.append(path)

if not output_candidates:
    output_candidates=list((BASE/"output").rglob("*.mp4"))
if not output_candidates:
    output_candidates=list((BASE/"output").rglob("*.webm"))
if not output_candidates:
    print(json.dumps(history.get("outputs",{}),ensure_ascii=False)[:8000])
    print(LOG.read_text(errors="replace")[-6000:])
    raise RuntimeError("工作流程已完成，但找不到產生的影片檔案")

video_path=max(output_candidates,key=lambda p:p.stat().st_mtime)
print("🎉 生成完成：",video_path)
print("大小：",round(video_path.stat().st_size/1024**2,1),"MiB")

if DRIVE_OUTPUT is not None:
    target=DRIVE_OUTPUT/f"{video_path.stem}_{int(time.time())}{video_path.suffix}"
    shutil.copy2(video_path,target)
    print("✅ 已備份至 Google Drive：",target)

display(Video(filename=str(video_path),embed=True,width=480))


## 常見問題
- **403**：本筆記本不開網頁介面，使用本機 ComfyUI API，避開先前瀏覽器的 8188 授權問題。
- **GPU OOM**：優先改小畫布（例如 384×672，皆為 32 的倍數）或改用 A100；若仍失敗，需增加 CPU RAM/使用更激進的卸載。即使 L4 顯存達 24GB，仍不保證可完成所有配置。
- **硬碟不足**：需要重新選執行階段或清理舊模型。Drive 僅保留最終影片，模型預設在 Colab 暫存。
- **模型授權**：[MiniMax H3 開源權重說明](https://github.com/MiniMax-AI/MiniMax-H3)。
- **官方工作流程**：[ComfyUI H3 I2V](https://docs.comfy.org/tutorials/video/minimax/minimax-h3-native)。
- 注意：長時間 GPU 運算仍受 Colab 配額及中斷影響。此程式碼經結構設計，未在你的執行階段端到端推論測試。
